In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import scipy.io as sio
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

def clean_run_data(raw_data, subject_id, run_type):
    """
    Cleans a single matrix of trials and formats it into a DataFrame.
    """
    # Define the columns based on the dataset description
    columns = ['r_imm', 'r_del', 'delay', 'choice', 'p_imm', 'condition', 'rt']
    df = pd.DataFrame(raw_data[:, :7], columns=columns)

    # Add tracking columns
    df['subject_id'] = subject_id
    df['run_type'] = run_type

    # Clean the data
    # Remove missing choices (Choice == 0)
    df = df[df['choice'] != 0].copy()

    # Recode choices: 1 = immediate, 2 = delayed -> convert to 1 (imm) and 0 (del)
    df['choice'] = df['choice'].apply(lambda x: 1 if x == 1 else 0)

    # Handle RTs: Drop impossible RTs (e.g., <= 0) and take log
    df = df[df['rt'] > 0].copy()
    df['log_rt'] = np.log(df['rt'])

    return df

def compile_master_dataset(input_folder, output_filepath):
    """
    Iterates through all .mat files in a folder and compiles them into one DataFrame.
    """
    all_data_frames = []

    # Find all .mat files in the specified folder
    mat_files = glob.glob(os.path.join(input_folder, '*.mat'))

    print(f"Found {len(mat_files)} participant files. Processing...")

    for idx, filepath in enumerate(mat_files):
        # Create a unique subject ID (e.g., "Subj_01")
        subject_id = f"Subj_{idx + 1:02d}"

        # Load the MATLAB file
        try:
            mat_data = sio.loadmat(filepath)

            # Extract and clean data_train (Run A)
            if 'data_train' in mat_data:
                df_train = clean_run_data(mat_data['data_train'], subject_id, 'train')
                all_data_frames.append(df_train)

            # Extract and clean data_test (Run B)
            if 'data_test' in mat_data:
                df_test = clean_run_data(mat_data['data_test'], subject_id, 'test')
                all_data_frames.append(df_test)

        except Exception as e:
            print(f"Error processing {filepath}: {e}")

    # Concatenate all individual DataFrames into one master DataFrame
    master_df = pd.concat(all_data_frames, ignore_index=True)

    # Save to CSV
    master_df.to_csv(output_filepath, index=False)
    print(f"Successfully saved master dataset to: {output_filepath}")
    print(f"Total rows: {len(master_df)}")

    return master_df

# ==========================================
# EXECUTION BLOCK
# ==========================================
# Update this path to wherever you unzipped the "DD data" folder in your Drive
INPUT_FOLDER = '/content/drive/MyDrive/DD_data/'

# Define where you want the compiled CSV to be saved
OUTPUT_FILE = '/content/drive/MyDrive/DD_master_dataset.csv'

# Run the compilation
master_data = compile_master_dataset(INPUT_FOLDER, OUTPUT_FILE)

# Display a preview of the structured data
display(master_data.head())

Mounted at /content/drive
Found 99 participant files. Processing...
Successfully saved master dataset to: /content/drive/MyDrive/DD_master_dataset.csv
Total rows: 28897


,r_imm,r_del,delay,choice,p_imm,condition,rt,subject_id,run_type,log_rt
0,0.26,5.0,30.0,0,0.5,1.0,2706.1,Subj_01,train,7.903264
1,9.93,10.0,7.0,1,0.5,1.0,3778.1,Subj_01,train,8.236977
2,19.42,20.0,30.0,1,0.5,1.0,4410.5,Subj_01,train,8.391743
3,14.65,20.0,365.0,1,0.5,1.0,3108.8,Subj_01,train,8.041992
4,4.99,5.0,180.0,1,0.5,1.0,2452.6,Subj_01,train,7.804904
